# D2-05 Regionalised methods across impact categories

This notebook now uses a **real** `edges` example built from the BAFU database. We keep one familiar product system, then compare how two regionalized impact categories read that same inventory differently.


## Learning goals

After this notebook, you should be able to:
- run two real `edges` methods on the same BAFU activity
- explain why raw scores from different impact categories should not be compared directly
- identify which exchanges dominate each category on one shared product system
- interpret why a water-scarcity result and a biodiversity result can highlight different hotspots


## Background references

- Sacchi, R., Menacho, A. H., Seitfudem, G., Agez, M., Schlesinger-Martinat, J., Koyamparambath, A., Saldivar, J. S., Loubet, P., & Bauer, C. (2025). Contextual LCIA without the overhead: an exchange-based framework for flexible impact assessment. *The International Journal of Life Cycle Assessment, 30*(12), 3087-3101. https://doi.org/10.1007/s11367-025-02551-7
- Seitfudem, G., Berger, M., Schmied, H. M., & Boulay, A. (2025). The updated and improved method for water scarcity impact assessment in LCA, AWARE2.0. *Journal of Industrial Ecology, 29*(3), 891-907. https://doi.org/10.1111/jiec.70023
- Schipper, A. M., et al. (2025). Intactness-based biodiversity impact factors for life cycle assessment. *Scientific Data*. https://doi.org/10.1038/s41597-025-05946-1


## 1) Select one familiar BAFU activity

We reuse a familiar Day 1 example: passenger-car operation with petrol. The idea is simple: keep the **inventory** fixed, and let the **impact category** change.


In [ ]:
import io
from contextlib import redirect_stderr, redirect_stdout

import country_converter as coco
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px

import bw2data as bd
from edges import EdgeLCIA

pd.set_option('display.max_colwidth', None)

bd.projects.set_current('barcelona-rlcia-2026')
db = bd.Database('bafu-2025') if 'bafu-2025' in bd.databases else bd.Database('bafu')

search_hits = [
    act for act in db.search('passenger car petrol')
    if act.get('location') == 'CH'
][:10]

pd.DataFrame([
    {
        'name': act['name'],
        'location': act.get('location'),
        'unit': act.get('unit'),
    }
    for act in search_hits
])


In [ ]:
activity = next(
    act for act in db
    if act['name'] == 'Operation, passenger car, petrol, 15% vol. ETBE with ethanol from biomass, EURO4'
    and act.get('location') == 'CH'
)

pd.DataFrame([
    {
        'database': activity.key[0],
        'code': activity.key[1],
        'name': activity['name'],
        'location': activity.get('location'),
        'reference product': activity.get('reference product'),
        'unit': activity.get('unit'),
    }
])


## 2) Define two real `edges` methods

We will compare:
- `AWARE 2.0`: a regionalized water-scarcity method
- `IBIF all pressures`: a regionalized biodiversity method that combines several pressure types

The raw units differ, so later we will compare **contribution patterns**, not raw score magnitudes.


In [ ]:
aware_method = ('AWARE 2.0', 'Country', 'all', 'yearly')
ibif_method = ('IBIF', 'biodiversity', 'all pressures', 'overall')

methods_df = pd.DataFrame([
    {
        'label': 'AWARE 2.0 water scarcity',
        'method tuple': aware_method,
        'focus': 'regionalized water scarcity / availability',
    },
    {
        'label': 'IBIF biodiversity (all pressures)',
        'method tuple': ibif_method,
        'focus': 'regionalized biodiversity damage across multiple pressures',
    },
])
methods_df


## 3) Run the full `edges` workflow once per category

The helper function below keeps the repeated `edges` steps together so the notebook stays readable.


In [ ]:
def run_edges_method(activity, method):
    lca = EdgeLCIA({activity: 1}, method)
    lca.lci()
    lca.map_exchanges()
    lca.map_aggregate_locations()
    lca.map_dynamic_locations()
    lca.map_contained_locations()
    lca.map_remaining_locations_to_global()
    lca.evaluate_cfs()
    lca.lcia()

    cf_table = lca.generate_cf_table(include_unmatched=True)
    matched = cf_table[cf_table['CF'].notna()].copy()
    matched['abs_impact'] = matched['impact'].abs()

    return {
        'lca': lca,
        'cf_table': cf_table,
        'matched': matched,
        'score': lca.score,
    }


In [ ]:
results = {
    'AWARE 2.0 water scarcity': run_edges_method(activity, aware_method),
    'IBIF biodiversity (all pressures)': run_edges_method(activity, ibif_method),
}

summary = pd.DataFrame([
    {
        'category': label,
        'raw score': result['score'],
        'matched CF rows': len(result['matched']),
        'top supplier flow by absolute contribution': result['matched'].groupby('supplier name')['abs_impact'].sum().sort_values(ascending=False).index[0],
        'top consumer activity by absolute contribution': result['matched'].groupby('consumer name')['abs_impact'].sum().sort_values(ascending=False).index[0],
    }
    for label, result in results.items()
])
summary


Raw scores from different categories should **not** be compared directly. The units and meanings differ.

For this reason, the rest of the notebook compares **which exchanges dominate each category**, not which raw score is numerically larger.


## 4) Inspect the AWARE result before comparing categories

In this real example, the signed AWARE score is negative. That does **not** mean the activity is "good" in some simple sense. It means the matched water-related exchanges include both negative and positive characterized contributions.

So first, inspect the largest negative and positive AWARE rows.


In [ ]:
aware_matched = results['AWARE 2.0 water scarcity']['matched']
cols = ['supplier name', 'consumer name', 'consumer location', 'amount', 'CF', 'impact']

print('Largest negative AWARE rows')
display(aware_matched[cols].sort_values('impact').head(8))

print('Largest positive AWARE rows')
display(aware_matched[cols].sort_values('impact', ascending=False).head(8))


## 5) Compare dominant contributors across categories using absolute contributions

To compare category structure more fairly, we now use the **absolute characterized contribution** of each row.


In [ ]:
def top_contributors(result, group_col, n=8):
    return (
        result['matched']
        .groupby(group_col)['abs_impact']
        .sum()
        .sort_values(ascending=False)
        .head(n)
        .rename('absolute characterized contribution')
        .reset_index()
    )

print('Top supplier flows for AWARE 2.0')
display(top_contributors(results['AWARE 2.0 water scarcity'], 'supplier name'))

print('Top supplier flows for IBIF biodiversity')
display(top_contributors(results['IBIF biodiversity (all pressures)'], 'supplier name'))


In [ ]:
print('Top consumer activities for AWARE 2.0')
display(top_contributors(results['AWARE 2.0 water scarcity'], 'consumer name'))

print('Top consumer activities for IBIF biodiversity')
display(top_contributors(results['IBIF biodiversity (all pressures)'], 'consumer name'))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, (label, result) in zip(axes, results.items()):
    shares = (
        result['matched']
        .groupby('supplier name')['abs_impact']
        .sum()
        .sort_values(ascending=False)
    )
    shares = (shares.head(6) / shares.sum()).sort_values()

    ax.barh(shares.index, shares.values, color='#4C78A8')
    ax.set_title(label)
    ax.set_xlabel('Share of total absolute characterized impact')

plt.tight_layout()
plt.show()


## 6) Put the location pattern on a map

The tables above show dominant flows and activities. A map adds another view: **where** the matched consumer locations are concentrated.

Because the category units differ, the map uses each location's **share of the total absolute characterized impact** within its own category.


In [ ]:
def location_shares(result):
    shares = (
        result['matched']
        .groupby('consumer location')['abs_impact']
        .sum()
        .sort_values(ascending=False)
        .rename('absolute characterized contribution')
        .reset_index()
    )
    shares['share'] = shares['absolute characterized contribution'] / shares['absolute characterized contribution'].sum()
    shares['share_pct'] = 100 * shares['share']
    return shares


def to_iso3(location):
    sink = io.StringIO()
    with redirect_stdout(sink), redirect_stderr(sink):
        iso3 = coco.convert(names=location, to='ISO3', not_found=None)
    if not iso3 or iso3 == location or len(str(iso3)) != 3:
        return None
    return str(iso3)


def plot_location_share_map(location_df, title, color_scale):
    rows = []
    skipped = []
    for _, row in location_df.iterrows():
        iso3 = to_iso3(row['consumer location'])
        if iso3 is None:
            skipped.append(row['consumer location'])
            continue
        rows.append(
            {
                'location': row['consumer location'],
                'iso3': iso3,
                'share_pct': row['share_pct'],
            }
        )

    frame = pd.DataFrame(rows)
    fig = px.choropleth(
        frame,
        locations='iso3',
        color='share_pct',
        hover_name='location',
        projection='natural earth',
        color_continuous_scale=color_scale,
        title=title,
    )
    fig.update_geos(showcoastlines=True, showcountries=True, fitbounds='locations')
    fig.show()

    if skipped:
        print('Not shown on the country-level map:', skipped)


aware_location_shares = location_shares(results['AWARE 2.0 water scarcity'])
ibif_location_shares = location_shares(results['IBIF biodiversity (all pressures)'])

print('Largest country or region shares for AWARE 2.0')
display(aware_location_shares.head(10))

print('Largest country or region shares for IBIF biodiversity')
display(ibif_location_shares.head(10))


In [ ]:
plot_location_share_map(
    aware_location_shares,
    'AWARE 2.0: share of total absolute characterized impact by consumer location',
    'Blues',
)

plot_location_share_map(
    ibif_location_shares,
    'IBIF biodiversity: share of total absolute characterized impact by consumer location',
    'Greens',
)


## Checkpoint 1

Using the tables, plot, and maps above:
- Which supplier flow dominates the AWARE result?
- Which supplier flow dominates the IBIF result?
- Does the direct petrol-car operation remain the dominant **consumer activity** in both categories?
- Which category looks more concentrated in Switzerland itself on the country-level map?


In [ ]:
# TODO
# aware_top_supplier = ...
# ibif_top_supplier = ...
# aware_top_consumer = ...
# ibif_top_consumer = ...
# country_map_comment = ...


In [ ]:
aware_top_supplier = (
    results['AWARE 2.0 water scarcity']['matched']
    .groupby('supplier name')['abs_impact']
    .sum()
    .sort_values(ascending=False)
    .index[0]
)
ibif_top_supplier = (
    results['IBIF biodiversity (all pressures)']['matched']
    .groupby('supplier name')['abs_impact']
    .sum()
    .sort_values(ascending=False)
    .index[0]
)
aware_top_consumer = (
    results['AWARE 2.0 water scarcity']['matched']
    .groupby('consumer name')['abs_impact']
    .sum()
    .sort_values(ascending=False)
    .index[0]
)
ibif_top_consumer = (
    results['IBIF biodiversity (all pressures)']['matched']
    .groupby('consumer name')['abs_impact']
    .sum()
    .sort_values(ascending=False)
    .index[0]
)
country_map_comment = 'IBIF looks more concentrated in Switzerland itself, while AWARE is dominated more strongly by upstream regional electricity and fuel supply locations.'

print('AWARE dominant supplier flow:', aware_top_supplier)
print('IBIF dominant supplier flow:', ibif_top_supplier)
print('AWARE dominant consumer activity:', aware_top_consumer)
print('IBIF dominant consumer activity:', ibif_top_consumer)
print('Country-level map comment:', country_map_comment)


## Recap

After this notebook, you should now be able to:
- run two real regionalized `edges` methods on one BAFU activity
- explain why category scores with different units should not be compared directly
- inspect signed and absolute characterized contributions from `generate_cf_table()`
- identify how the dominant flows and dominant activities can change from water scarcity to biodiversity
- read a country-level map of regionalized contribution shares while recognizing that aggregate regions such as `RER` or `GLO` are not shown directly
